# E-commerce Sales & Profit Analytics using Data Cleaning and Visualization

**Internship submission:** AICTE | IBM SkillsBuild Data Analytics with AI — BharatCares  
**Prepared by:** Aarti Katkar

## Project overview
This project analyzes the supplied practice workbook `AI + Data_ Make Data Intelligent _ Masterclass 1 _ Practice Dataset.xlsx`. It inspects data quality, cleans inconsistent formats, creates business metrics, and explores revenue and profit by product category, region, and month.

**Important:** The workbook is a practice dataset. Treat results as analysis of the supplied data, not verified real-world business results. Data cleaning decisions are documented and rows are not silently overwritten in the original DataFrame.

## Run this notebook
- **Google Colab:** upload the Excel workbook when prompted.
- **Jupyter on Windows:** place the workbook in the same folder as this notebook.
- Run cells from top to bottom.


In [ ]:
import io
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


## 1. Load the supplied Excel workbook

The workbook should contain a sheet named `Raw_Data`. If using Google Colab, the upload dialog will appear. If running locally, the notebook looks for the workbook in the current folder.


In [ ]:
DATA_FILE = 'AI + Data_ Make Data Intelligent _ Masterclass 1 _ Practice Dataset.xlsx'

if not Path(DATA_FILE).exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        # Use the first uploaded Excel file
        candidates = [name for name in uploaded if name.lower().endswith(('.xlsx', '.xls'))]
        if not candidates:
            raise FileNotFoundError('Please upload the supplied .xlsx workbook.')
        DATA_FILE = candidates[0]
    except ImportError:
        raise FileNotFoundError(
            f'Could not find {DATA_FILE}. Place the supplied Excel workbook beside this notebook, '
            'or open this notebook in Google Colab and upload it.'
        )

raw = pd.read_excel(DATA_FILE, sheet_name='Raw_Data')
print(f'Loaded {raw.shape[0]:,} rows and {raw.shape[1]} columns.')
display(raw.head())


## 2. Understand the columns

- `Order_ID`: order identifier.
- `Order_Date`: date of the order.
- `Customer_ID`: customer identifier.
- `Product`: product name.
- `Category`: product category.
- `Region`: sales region/city.
- `Quantity`: units in the order.
- `Revenue`: order revenue, sometimes stored as text with currency labels.
- `Profit`: reported profit.

The `Prompts` sheet in the workbook contains practice prompts; the analysis uses `Raw_Data`.


In [ ]:
column_profile = pd.DataFrame({
    'Column': raw.columns,
    'Data type': [str(raw[c].dtype) for c in raw.columns],
    'Missing values': [int(raw[c].isna().sum()) for c in raw.columns],
    'Unique values': [int(raw[c].nunique(dropna=True)) for c in raw.columns]
})
display(column_profile)


## 3. Data-quality audit

The audit checks missing values, duplicate records/order IDs, inconsistent labels, invalid dates, currency-formatted revenue, and suspicious negative quantities. Review these findings before using the cleaned data.


In [ ]:
audit = {
    'Rows in source': len(raw),
    'Exact duplicate rows': int(raw.duplicated().sum()),
    'Duplicate Order_ID rows after first occurrence': int(raw['Order_ID'].duplicated().sum()),
    'Missing Customer_ID': int(raw['Customer_ID'].isna().sum()),
    'Missing Product': int(raw['Product'].isna().sum()),
    'Missing Region': int(raw['Region'].isna().sum()),
    'Missing Quantity': int(raw['Quantity'].isna().sum()),
    'Missing Revenue': int(raw['Revenue'].isna().sum()),
    'Missing Profit': int(raw['Profit'].isna().sum()),
    'Negative quantities': int((pd.to_numeric(raw['Quantity'], errors='coerce') < 0).sum()),
    'Revenue values containing non-numeric currency/text characters': int(raw['Revenue'].astype(str).str.contains(r'[^0-9.\\-]', regex=True).sum())
}
display(pd.DataFrame({'Check': audit.keys(), 'Count': audit.values()}))

print('Category labels before standardization:')
display(raw['Category'].value_counts(dropna=False).to_frame('rows'))
print('Region labels before standardization:')
display(raw['Region'].value_counts(dropna=False).to_frame('rows'))


## 4. Clean and standardize data

Cleaning choices:
1. Parse dates in ISO (`YYYY-MM-DD`) or day-first (`DD-MM-YYYY`) format; invalid dates become missing and are reported.
2. Convert revenue text to numeric by removing currency labels and symbols. Values such as `374814 INR` become `374814`.
3. Standardize category, region, and product text casing/spacing.
4. Remove duplicate `Order_ID` records after retaining the first occurrence.
5. Keep a separate cleaned table and flag negative quantities. Negative quantities are excluded from the main sales KPI dataset pending business review; they are not automatically changed to positive.
6. Keep rows with missing critical numeric fields out of KPI calculations, and report how many remain.


In [ ]:
clean = raw.copy()

def parse_order_date(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.to_datetime(value, errors='coerce')
    text = str(value).strip()
    # ISO-like dates are year-first; other text dates are day-first.
    if re.match(r'^\\d{4}[-/]\\d{1,2}[-/]\\d{1,2}', text):
        return pd.to_datetime(text, errors='coerce', yearfirst=True)
    return pd.to_datetime(text, errors='coerce', dayfirst=True)

def parse_money(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    text = re.sub(r'(?i)inr|rs\\.?|rupees?|₹|,|\\s', '', text)
    text = re.sub(r'[^0-9.\\-]', '', text)
    try:
        return float(text) if text not in ('', '.', '-', '-.') else np.nan
    except ValueError:
        return np.nan

clean['Order_Date'] = clean['Order_Date'].apply(parse_order_date)
for col in ['Revenue', 'Profit', 'Quantity']:
    clean[col] = pd.to_numeric(clean[col].apply(parse_money) if col == 'Revenue' else clean[col], errors='coerce')

for col in ['Product', 'Category', 'Region']:
    clean[col] = clean[col].astype('string').str.strip().str.title()
clean['Customer_ID'] = clean['Customer_ID'].astype('string').str.strip().replace('', pd.NA)

invalid_date_count = int(clean['Order_Date'].isna().sum())
negative_quantity_count = int((clean['Quantity'] < 0).sum())
duplicate_order_count = int(clean['Order_ID'].duplicated().sum())

# Deduplicate by order identifier; keep first occurrence and document this choice.
clean = clean.drop_duplicates(subset=['Order_ID'], keep='first').copy()
clean['Month'] = clean['Order_Date'].dt.to_period('M').astype('string')
clean['Year'] = clean['Order_Date'].dt.year

# Exclude suspicious negative quantities and rows missing fields needed for sales KPIs.
analysis = clean.loc[
    (clean['Quantity'] >= 0) &
    clean['Revenue'].notna() &
    clean['Profit'].notna() &
    clean['Quantity'].notna() &
    clean['Order_Date'].notna()
].copy()

print('Invalid/missing dates after parsing:', invalid_date_count)
print('Negative quantity rows flagged:', negative_quantity_count)
print('Duplicate Order_ID rows removed:', duplicate_order_count)
print('Rows after de-duplication:', len(clean))
print('Rows available for KPI analysis:', len(analysis))
display(analysis.head())


## 5. Create business KPIs

KPIs are calculated only from rows that pass the documented cleaning filters. `Revenue` and `Profit` are treated as currency amounts as provided by the practice dataset; the workbook does not establish a verified currency standard for every row.


In [ ]:
total_revenue = analysis['Revenue'].sum()
total_profit = analysis['Profit'].sum()
total_units = analysis['Quantity'].sum()
order_count = analysis['Order_ID'].nunique()
customer_count = analysis['Customer_ID'].nunique()
profit_margin = (total_profit / total_revenue * 100) if total_revenue else np.nan
average_order_revenue = (total_revenue / order_count) if order_count else np.nan

kpis = pd.DataFrame({
    'KPI': ['Total revenue', 'Total profit', 'Units sold', 'Orders analyzed',
            'Unique customers with ID', 'Profit margin (%)', 'Average revenue per order'],
    'Value': [total_revenue, total_profit, total_units, order_count,
              customer_count, profit_margin, average_order_revenue]
})
display(kpis.round(2))


## 6. Exploratory analysis and visualizations

The following charts compare category and region performance and show monthly trends. They are descriptive summaries, not proof of causation.


In [ ]:
category_summary = analysis.groupby('Category', dropna=False).agg(
    Orders=('Order_ID', 'nunique'),
    Units=('Quantity', 'sum'),
    Revenue=('Revenue', 'sum'),
    Profit=('Profit', 'sum')
).sort_values('Revenue', ascending=False)
category_summary['Profit_Margin_%'] = np.where(
    category_summary['Revenue'] != 0,
    category_summary['Profit'] / category_summary['Revenue'] * 100,
    np.nan
)
display(category_summary.round(2))

ax = category_summary['Revenue'].sort_values().plot(kind='barh', figsize=(9, 5), title='Revenue by Product Category')
ax.set_xlabel('Revenue')
plt.tight_layout()
plt.show()

region_summary = analysis.groupby('Region', dropna=False).agg(
    Orders=('Order_ID', 'nunique'),
    Revenue=('Revenue', 'sum'),
    Profit=('Profit', 'sum')
).sort_values('Revenue', ascending=False)
display(region_summary.round(2))

ax = region_summary['Revenue'].sort_values().plot(kind='barh', figsize=(9, 5), title='Revenue by Region')
ax.set_xlabel('Revenue')
plt.tight_layout()
plt.show()

monthly = analysis.dropna(subset=['Order_Date']).groupby(analysis['Order_Date'].dt.to_period('M')).agg(
    Revenue=('Revenue', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order_ID', 'nunique')
)
monthly.index = monthly.index.astype(str)
display(monthly)

ax = monthly[['Revenue', 'Profit']].plot(figsize=(10, 5), marker='o', title='Monthly Revenue and Profit')
ax.set_xlabel('Month')
ax.set_ylabel('Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Product and customer analysis

This section identifies products with the highest aggregate revenue and customers with repeat orders. Customer IDs are identifiers and should be handled carefully; avoid publishing identifiable customer-level outputs.


In [ ]:
product_summary = analysis.groupby('Product').agg(
    Orders=('Order_ID', 'nunique'),
    Units=('Quantity', 'sum'),
    Revenue=('Revenue', 'sum'),
    Profit=('Profit', 'sum')
).sort_values('Revenue', ascending=False)
display(product_summary.head(10).round(2))

customer_orders = analysis.dropna(subset=['Customer_ID']).groupby('Customer_ID')['Order_ID'].nunique()
customer_segments = pd.DataFrame({
    'Orders': customer_orders,
    'Customer_Segment': np.where(customer_orders > 1, 'Repeat Customer', 'New Customer')
})
print('Customer segment counts:')
display(customer_segments['Customer_Segment'].value_counts().to_frame('Customers'))


## 8. Export cleaned data and summary tables

These output files can be used for further practice in Excel, Power BI, or Google Sheets. Download links are offered when running in Google Colab.


In [ ]:
output_dir = Path('project_outputs')
output_dir.mkdir(exist_ok=True)

analysis.to_csv(output_dir / 'cleaned_ecommerce_data.csv', index=False)
category_summary.to_csv(output_dir / 'category_summary.csv')
region_summary.to_csv(output_dir / 'region_summary.csv')
monthly.to_csv(output_dir / 'monthly_summary.csv', index=True)
kpis.to_csv(output_dir / 'business_kpis.csv', index=False)

print('Saved files in:', output_dir.resolve())
try:
    from google.colab import files
    for filename in ['cleaned_ecommerce_data.csv', 'category_summary.csv', 'region_summary.csv', 'monthly_summary.csv', 'business_kpis.csv']:
        files.download(str(output_dir / filename))
except ImportError:
    print('Files are saved locally in the project_outputs folder.')


## 9. Business questions this analysis can support

1. Which product category generates the highest revenue and profit?
2. Which region contributes the most revenue?
3. How do revenue and profit change month by month?
4. Which products contribute the most revenue?
5. Which customers appear as repeat customers in this dataset?
6. Which regions have a comparatively low profit margin?
7. How many records were excluded due to missing numeric values, invalid dates, or negative quantities?
8. Which data-quality issues should be reviewed before using this dataset for decisions?

## 10. Conclusion and limitations

This project demonstrates a repeatable analytics workflow: data profiling, quality checks, cleaning, KPI calculation, grouping, and visualization. Results depend on the supplied workbook and the cleaning rules documented above. Negative quantities may represent returns or data errors; they are flagged and excluded from primary sales KPIs pending confirmation. Invalid dates and missing values are reported. The workbook appears to be a practice dataset, so findings should not be presented as verified business performance.

### References
- Supplied workbook: `AI + Data_ Make Data Intelligent _ Masterclass 1 _ Practice Dataset.xlsx`
- pandas documentation: https://pandas.pydata.org/docs/
- Matplotlib documentation: https://matplotlib.org/stable/
